# Autoencoders

An autoencoder is an **unsupervised** neural network architecture that learns to compress data into a lower-dimensional representation and then reconstruct it back to its original form. The fundamental goal is not simply to copy the input to the output, but to force the network to prioritize which properties of the data are most important, thereby learning a meaningful feature representation.

## The Mathematical Formulation of Autoencoders

A standard autoencoder consists of two parameterized functions, the encoder $f_\phi$ and the decoder $g_\theta$. Given an input vector $x \in \mathbb{R}^d$, the encoder maps it to a **latent representation** or **bottleneck** $z \in \mathbb{R}^k$:

$$z = f_\phi(x) = \sigma(W x + b)$$

Here, $W$ is the weight matrix, $b$ is the bias vector, and $\sigma$ is a non-linear activation function such as ReLU or Sigmoid. The decoder then maps this latent representation back to the original data space to produce a reconstruction $\tilde{x} \in \mathbb{R}^d$:

$$\tilde{x} = g_\theta(z) = \sigma(W' z + b')$$

The network is trained to minimize a loss function $\mathcal{L}$ that measures the divergence between the original input and the reconstruction. For continuous data, this is typically the Mean Squared Error (MSE):

$$\mathcal{L}(\theta, \phi) = \frac{1}{N} \sum_{i=1}^N \Vert{}x_i - g_\theta(f_\phi(x_i))\Vert{}^2$$

In [13]:
import os

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [4]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing researcher workflow on device: {device}")

Executing researcher workflow on device: mps


In [14]:
class SimpleAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim),
            nn.Sigmoid() 
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed, latent

In [15]:
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000, shuffle=False, num_workers=2)

In [16]:
input_size = 784
bottleneck_size = 32
model = SimpleAutoencoder(input_size, bottleneck_size).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

In [17]:
epochs = 50
patience = 5
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_path = 'models/autoencoder-best.pth'

print("Beginning training...")
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    for images, _ in train_loader:
        images = images.view(images.size(0), -1).to(device)
        reconstructed, _ = model(images)
        loss = criterion(reconstructed, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_loader)
    
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, _ in val_loader:
            images = images.view(images.size(0), -1).to(device)
            reconstructed, _ = model(images)
            loss = criterion(reconstructed, images)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    scheduler.step(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"  -> Validation loss improved. Model saved to {best_model_path}")
    else:
        epochs_no_improve += 1
        print(f"  -> No improvement in validation loss for {epochs_no_improve} epoch(s).")
        if epochs_no_improve >= patience:
            print("Early stopping triggered. Training stopped.")
            break

if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path))
    print("Loaded the best model weights for evaluation.")

Beginning training...
Epoch [1/50] | Train Loss: 0.0644 | Val Loss: 0.0358
  -> Validation loss improved. Model saved to models/autoencoder-best.pth
Epoch [2/50] | Train Loss: 0.0293 | Val Loss: 0.0237
  -> Validation loss improved. Model saved to models/autoencoder-best.pth
Epoch [3/50] | Train Loss: 0.0213 | Val Loss: 0.0183
  -> Validation loss improved. Model saved to models/autoencoder-best.pth
Epoch [4/50] | Train Loss: 0.0173 | Val Loss: 0.0157
  -> Validation loss improved. Model saved to models/autoencoder-best.pth
Epoch [5/50] | Train Loss: 0.0152 | Val Loss: 0.0141
  -> Validation loss improved. Model saved to models/autoencoder-best.pth
Epoch [6/50] | Train Loss: 0.0138 | Val Loss: 0.0130
  -> Validation loss improved. Model saved to models/autoencoder-best.pth
Epoch [7/50] | Train Loss: 0.0128 | Val Loss: 0.0122
  -> Validation loss improved. Model saved to models/autoencoder-best.pth
Epoch [8/50] | Train Loss: 0.0119 | Val Loss: 0.0112
  -> Validation loss improved. Model

## The Problem of Informational Collapse

If the latent dimension $k$ is equal to or greater than the input dimension $d$, the network has excessive information capacity. In this scenario, the autoencoder is susceptible to informational collapse, where it simply learns the identity function ($f(x) = x$). When this happens, the reconstruction error becomes zero over the entire space, and the latent vector $z$ fails to capture any useful, abstracted features of the data distribution. To prevent collapse, autoencoders must be **constrained**. The most basic constraint is creating an undercomplete autoencoder, where $k < d$. This forces the network to perform dimensionality reduction, conceptually similar to Principal Component Analysis (PCA) if the activation functions are strictly linear.

## Regularized Architectures

When a larger latent capacity is required, regularization techniques are introduced to prevent the model from learning the identity function.

### Sparse Autoencoders

A sparsity penalty (typically an $L_1$ norm) is added to the loss function, forcing the majority of the latent activations in $z$ to remain inactive (zero) for any given input.

$$\mathcal{L} = \Vert{}x - \tilde{x}\Vert{}^2 + \lambda \sum_{j=1}^k \vert{}z_j\vert{}$$

When you train a network, you update the latent variable $z$ by calculating the derivative (the slope) of your loss function and subtracting it. Your loss function with $L_1$ regularization is:$$Loss = \text{Reconstruction Error} + \lambda \vert{}z\vert{}$$To update $z$, the neural network takes the derivative of that equation. This is where the absolute mathematical difference between $L_1$ and $L_2$ happens. The derivative of the $L_2$ penalty ($z^2$) is $2z$. The derivative of the $L_1$ penalty ($\vert{}z\vert{}$) is a constant $+1$ (if $z$ is positive) or $-1$ (if $z$ is negative). Now plug that into the standard Gradient Descent update rule:

$$z_{new} = z_{old} - (\text{learning\_rate} \times \text{slope})$$

If you are using $L_2$, and $z$ drops to $0.001$, the slope ($2z$) becomes $0.002$. The update step becomes microscopic. The variable $z$ glides infinitely closer to zero but never mathematically reaches it, like Zeno's paradox. It just gets smaller and smaller forever. If you are using $L_1$, and $z$ drops to $0.001$, the slope of $\vert{}z\vert{}$ is still exactly $1$. The update step does not shrink. The network subtracts the exact same large chunk from $z$ that it would if $z$ were $10,000$.

Because it takes a full-sized step, $z$ instantly overshoots zero and goes negative. The moment it goes negative, the derivative of $\vert{}z\vert{}$ instantly snaps to $-1$. On the very next training step, the network adds that full-sized chunk right back, throwing $z$ positive again. The variable $z$ violently ricochets back and forth across zero with every single training step. Modern optimization algorithms (using subgradient methods or proximal gradient descent) are programmed to recognize this exact oscillation. When they see a variable bouncing back and forth across zero because the derivative won't shrink, the optimizer clamps the value at exactly $0$ to stabilize the network. $L_1$ zeroes out variables because the derivative of an absolute value never tapers off, forcing the optimizer to endlessly subtract a constant chunk until it crashes into zero and gets clamped.

In [18]:
class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
            nn.ReLU() # ReLU applied here to enforce sparsity
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim),
            nn.Sigmoid() 
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed, latent

In [19]:
model_sae = SparseAutoencoder(input_size, bottleneck_size).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model_sae.parameters(), lr=1e-3)

# Hyperparameter for L1 penalty
sparsity_weight = 1e-3

In [20]:
print("Beginning SAE training...")
for epoch in range(epochs):
    model_sae.train()
    running_loss = 0.0
    
    for images, _ in train_loader:
        images = images.view(images.size(0), -1).to(device)
        reconstructed, latent = model_sae(images)

        # Loss = MSE + L1 Regularization on latent activations
        mse_loss = criterion(reconstructed, images)
        l1_loss = torch.mean(torch.abs(latent))
        loss = mse_loss + sparsity_weight * l1_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{epochs}], SAE Loss (MSE+L1): {running_loss/len(train_loader):.4f}")

Beginning SAE training...
Epoch [1/50], SAE Loss (MSE+L1): 0.0675
Epoch [2/50], SAE Loss (MSE+L1): 0.0330
Epoch [3/50], SAE Loss (MSE+L1): 0.0248
Epoch [4/50], SAE Loss (MSE+L1): 0.0211
Epoch [5/50], SAE Loss (MSE+L1): 0.0187
Epoch [6/50], SAE Loss (MSE+L1): 0.0173
Epoch [7/50], SAE Loss (MSE+L1): 0.0165
Epoch [8/50], SAE Loss (MSE+L1): 0.0159
Epoch [9/50], SAE Loss (MSE+L1): 0.0153
Epoch [10/50], SAE Loss (MSE+L1): 0.0149
Epoch [11/50], SAE Loss (MSE+L1): 0.0145
Epoch [12/50], SAE Loss (MSE+L1): 0.0142
Epoch [13/50], SAE Loss (MSE+L1): 0.0139
Epoch [14/50], SAE Loss (MSE+L1): 0.0136
Epoch [15/50], SAE Loss (MSE+L1): 0.0134
Epoch [16/50], SAE Loss (MSE+L1): 0.0132
Epoch [17/50], SAE Loss (MSE+L1): 0.0130
Epoch [18/50], SAE Loss (MSE+L1): 0.0128
Epoch [19/50], SAE Loss (MSE+L1): 0.0127
Epoch [20/50], SAE Loss (MSE+L1): 0.0125
Epoch [21/50], SAE Loss (MSE+L1): 0.0124
Epoch [22/50], SAE Loss (MSE+L1): 0.0122
Epoch [23/50], SAE Loss (MSE+L1): 0.0121
Epoch [24/50], SAE Loss (MSE+L1): 0.0120

### Variational Autoencoders (VAE)

In a standard autoencoder, the final layer of the encoder outputs a single vector $z \in \mathbb{R}^k$. In a VAE, the final layer of the encoder splits into two parallel fully-connected layers. One layer outputs a vector of means $\mu \in \mathbb{R}^k$. The other layer outputs a vector of log-variances $\log(\sigma^2) \in \mathbb{R}^k$. The network outputs log-variance rather than variance for numerical stability, ensuring $\sigma^2$ is always positive when exponentiated. This means each of the $k$ independent latent dimensions gets its own specific mean and variance.

To pass data to the decoder, a latent vector $z$ must be sampled from the learned distribution $\mathcal{N}(\mu, \sigma^2)$. Sampling $z$ directly creates a bottleneck. You cannot backpropagate a gradient through a stochastic sampling operation. If gradients cannot flow backward from the decoder through the random sample to the encoder, the encoder's weights cannot be updated.

During training, the network calculates the total loss $\mathcal{L}$ at the very end of the decoder. To update the weights inside the encoder that produced the mean $\mu$, the network must calculate the gradient (derivative) of the loss with respect to $\mu$. By the Chain Rule, that gradient is:

$$\frac{\partial \mathcal{L}}{\partial \mu} = \frac{\partial \mathcal{L}}{\partial z} \cdot \frac{\partial z}{\partial \mu}$$

The decoder easily calculates $\frac{\partial \mathcal{L}}{\partial z}$. The problem is the second term: $\frac{\partial z}{\partial \mu}$. If you generate $z$ by telling a computer program to draw a random sample—$z \sim \mathcal{N}(\mu, \sigma^2)$—that operation is a stochastic process. It is an algorithmic dice roll, not a continuous mathematical equation. Because there is no equation, the derivative $\frac{\partial z}{\partial \mu}$ is mathematically undefined. If $\frac{\partial z}{\partial \mu}$ is undefined, the entire Chain Rule collapses to zero or an error. The gradient cannot flow backward through the dice roll, meaning the encoder never receives the feedback it needs to update its weights.

**Reparameterization trick** replaces the black-box dice roll with a differentiable equation:

$$z = \mu + \sigma \odot \epsilon$$

Here, $\epsilon$ is a random number drawn from a standard normal distribution $\mathcal{N}(0, I)$. $\mathcal{N}(0, I)$ is the mathematical notation for a standard multivariate normal distribution—basically, the most generic, vanilla bell curve in existence, stretched across multiple dimensions.Let's break down the exact symbols:

* $\mathcal{N}$: Stands for the Normal distribution (the bell curve).
* $0$: The mean. Because the latent space has $k$ dimensions, this isn't just a single zero; it is a vector of pure zeros (e.g., $[0, 0, 0, \dots]$). It means the exact center of this distribution sits permanently at zero.
* $I$: The Identity matrix. This represents the variance/covariance. An identity matrix is a grid of numbers with $1$s down the main diagonal and $0$s everywhere else. The $1$s mean every single dimension has a variance of exactly $1$. The $0$s mean the dimensions are completely independent—a random roll in dimension $A$ has absolutely zero effect on the roll in dimension $B$.

Because $\epsilon$ is drawn before the equation is evaluated, it acts as a constant value (e.g., $0.74$) during the calculus step. Now, look at the partial derivatives of that exact equation. The derivative of $z$ with respect to $\mu$:

$$\frac{\partial z}{\partial \mu} = \frac{\partial}{\partial \mu} (\mu + \sigma \odot \epsilon)$$

Because $\sigma$ and $\epsilon$ are treated as constants relative to $\mu$, their derivative is $0$. The derivative of $\mu$ is $1$.$$\frac{\partial z}{\partial \mu} = 1$$ The derivative of $z$ with respect to $\sigma$:

$$\frac{\partial z}{\partial \sigma} = \frac{\partial}{\partial \sigma} (\mu + \sigma \odot \epsilon)$$

Because $\mu$ is a constant relative to $\sigma$, its derivative is $0$. The derivative of $\sigma \odot \epsilon$ is simply the constant multiplier $\epsilon$.$$\frac{\partial z}{\partial \sigma} = \epsilon$$

Now substitute those solvable derivatives back into the original Chain Rule. To update the encoder weights responsible for the mean:$$\frac{\partial \mathcal{L}}{\partial \mu} = \frac{\partial \mathcal{L}}{\partial z} \cdot 1$$To update the encoder weights responsible for the variance:

$$\frac{\partial \mathcal{L}}{\partial \sigma} = \frac{\partial \mathcal{L}}{\partial z} \cdot \epsilon$$

By rewriting the sampling process as simple addition and multiplication, the unpredictable randomness ($\epsilon$) gets pushed to the side as a static multiplier. The network can execute the Chain Rule perfectly, gradients flow all the way back to the beginning of the encoder, and the weights update.

If the network solely optimized for reconstruction, it would set the variance $\sigma^2$ to exactly $0$ for every dimension, eliminating the noise entirely, and spread the $\mu$ vectors infinitely far apart so they never overlap. This collapses the model back into a standard, deterministic autoencoder with a discontinuous latent space. To prevent this, the loss function incorporates the **Kullback-Leibler (KL) Divergence**, which measures the difference between the network's parameterized distribution $\mathcal{N}(\mu, \sigma^2)$ and a standard multivariate normal distribution $\mathcal{N}(0, I)$. Because the covariance matrix is diagonal (dimensions are independent), the KL divergence formula simplifies analytically to:

$$D_{KL} = -\frac{1}{2} \sum_{i=1}^k \left( 1 + \log(\sigma_i^2) - \mu_i^2 - \sigma_i^2 \right)$$

This equation imposes mathematical constraints on the network. To minimize the $-\mu_i^2$ term, the network must push every $\mu_i$ toward $0$. To minimize the $(1 + \log(\sigma_i^2) - \sigma_i^2)$ term, the network must push every $\sigma_i^2$ toward $1$. The KL divergence acts as a regularizer that forces every encoded distribution to center around the origin of the latent space with a variance of $1$ across all dimensions.

#### ELBO Loss

The network is trained to maximize the Evidence Lower Bound (ELBO), which equates to minimizing this combined loss function:

$$\mathcal{L} = \text{Reconstruction\_Loss}(x, g_\theta(z)) + \beta D_{KL}(\mathcal{N}(\mu, \text{diag}(\sigma^2)) \parallel \mathcal{N}(0, I))$$

This establishes a strict mathematical tradeoff. Reconstruction Loss forces the network to minimize variance and segregate distributions so the decoder receives precise, distinct coordinates for $x$. KL Divergence forces the network to center all $\mu$ at $0$ and expand all $\sigma$ to $1$, driving the distributions to heavily overlap at the origin. This tradeoff forces the network to pack all latent representations into a dense, continuous region. Because the probability densities overlap, the decoder is forced to map the transitions between them smoothly. This creates a continuous, generative latent space where sampling a random vector $z$ near the origin will decode into a structurally valid output.

In [22]:
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        
        # Encoder
        self.encoder_fc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU()
        )
        
        # Latent distribution parameters
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim),
            nn.Sigmoid() 
        )

    def encode(self, x):
        h = self.encoder_fc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar) # logvar = log(sigma^2) => sigma = 0.5 * logvar
        eps = torch.randn_like(std)   # Random noise
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstructed = self.decoder(z)
        return reconstructed, mu, logvar

In [23]:
# VAE Custom Loss: Reconstruction (MSE) + KL Divergence
def vae_loss(recon_x, x, mu, logvar):
    MSE = nn.MSELoss(reduction='sum')(recon_x, x)
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (MSE + KLD) / x.size(0) # Average over batch

model_vae = VAE(input_size, bottleneck_size).to(device)
optimizer = optim.Adam(model_vae.parameters(), lr=1e-3)

In [24]:
print("Beginning VAE training...")
for epoch in range(epochs):
    model_vae.train()
    
    running_loss = 0.0
    for images, _ in train_loader:
        images = images.view(images.size(0), -1).to(device)
        reconstructed, mu, logvar = model_vae(images)
        loss = vae_loss(reconstructed, images, mu, logvar)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{epochs}], VAE Loss: {running_loss/len(train_loader):.4f}")

Beginning VAE training...
Epoch [1/50], VAE Loss: 62.2379
Epoch [2/50], VAE Loss: 43.3696
Epoch [3/50], VAE Loss: 39.2163
Epoch [4/50], VAE Loss: 36.8943
Epoch [5/50], VAE Loss: 35.5980
Epoch [6/50], VAE Loss: 34.7878
Epoch [7/50], VAE Loss: 34.1471
Epoch [8/50], VAE Loss: 33.6210
Epoch [9/50], VAE Loss: 33.2191
Epoch [10/50], VAE Loss: 32.9124
Epoch [11/50], VAE Loss: 32.6398
Epoch [12/50], VAE Loss: 32.4228
Epoch [13/50], VAE Loss: 32.2096
Epoch [14/50], VAE Loss: 32.0909
Epoch [15/50], VAE Loss: 31.9634
Epoch [16/50], VAE Loss: 31.8133
Epoch [17/50], VAE Loss: 31.7028
Epoch [18/50], VAE Loss: 31.6135
Epoch [19/50], VAE Loss: 31.5248
Epoch [20/50], VAE Loss: 31.4357
Epoch [21/50], VAE Loss: 31.3724
Epoch [22/50], VAE Loss: 31.2991
Epoch [23/50], VAE Loss: 31.2525
Epoch [24/50], VAE Loss: 31.1773
Epoch [25/50], VAE Loss: 31.1462
Epoch [26/50], VAE Loss: 31.0499
Epoch [27/50], VAE Loss: 31.0284
Epoch [28/50], VAE Loss: 30.9945
Epoch [29/50], VAE Loss: 30.9480
Epoch [30/50], VAE Loss: 3

### Denoising Autoencoders (DAE)

The input $x$ is intentionally corrupted by a noise process (e.g., masking pixels or adding Gaussian noise) to produce a contrastive sample $\hat{x}$. The network is then trained to reconstruct the original, clean $x$ from the corrupted $\hat{x}$. This forces the encoder to capture the underlying statistical dependencies of the data rather than memorizing exact values.

$$\mathcal{L} = \Vert{}x - g_\theta(f_\phi(\hat{x}))\Vert{}^2$$

In [25]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim),
            nn.Sigmoid() 
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed, latent

In [26]:
model_dae = DenoisingAutoencoder(input_size, bottleneck_size).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model_dae.parameters(), lr=1e-3)
noise_factor = 0.3 # How much noise to add

In [ ]:
print("Beginning DAE training...")
for epoch in range(epochs):
    model_dae.train()
    running_loss = 0.0
    
    for images, _ in train_loader:
        images = images.view(images.size(0), -1).to(device)
        
        # Inject Gaussian Noise and clip to valid pixel range [0, 1]
        noisy_images = images + noise_factor * torch.randn_like(images)
        noisy_images = torch.clamp(noisy_images, 0., 1.)
        # Feed the NOISY image in
        reconstructed, _ = model_dae(noisy_images)
        
        # Loss is computed against the CLEAN original image
        loss = criterion(reconstructed, images)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{epochs}], DAE Loss: {running_loss/len(train_loader):.4f}")

Beginning DAE training...
Epoch [1/50], DAE Loss: 0.0649
Epoch [2/50], DAE Loss: 0.0344
Epoch [3/50], DAE Loss: 0.0271
Epoch [4/50], DAE Loss: 0.0233
Epoch [5/50], DAE Loss: 0.0210
Epoch [6/50], DAE Loss: 0.0192



### The Energy-Based Perspective

In theoretical machine learning, autoencoders can be viewed as Energy-Based Models (EBMs). The scalar energy function $F_w(x)$ evaluates the degree of compatibility of the data. For an autoencoder, the energy is defined as the reconstruction error:

$$F_w(x) = D(x, g_\theta(f_\phi(x)))$$

Training an autoencoder is equivalent to shaping this energy landscape so that the model assigns low energy (low reconstruction error) to points on the true data manifold, and higher energy to points away from the data density. Denoising autoencoders explicitly do this by pushing up the energy of corrupted contrastive samples while minimizing the energy of the true data.